# Run Bradford Bulls Track-Level Annotation on Google Colab

Notebook này được thiết kế để chạy pipeline trên Google Colab do bạn không có môi trường GPU local. Nó sẽ tự động:
1. Clone repository từ Github về Colab
2. Cài đặt các thư viện cần thiết
3. Tải weights YOLO và chuẩn bị dữ liệu
4. Chạy pipeline trích xuất keyframe và clip

**Lưu ý cực kỳ quan trọng:** Bạn phải bật GPU trước khi chạy (Vào mục `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> Chọn `T4 GPU` hoặc các GPU khác).

## 1. Mount Google Drive (Khuyên dùng)
Mount Google Drive để bạn có thể copy video gốc từ Drive vào Colab, và copy kết quả (annotation package) ngược lại Drive mà không sợ bị mất dữ liệu khi Colab đóng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository và Cài đặt Dependencies

In [ ]:
# TODO: Thay đổi link github của bạn vào đây. 
# Nếu repo là private, bạn cần cấu hình Personal Access Token (PAT) trong link git, ví dụ: https://<PAT>@github.com/user/repo.git
GIT_REPO = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git"

# Xóa folder cũ nếu có để tránh lỗi clone lại
!rm -rf bradford_bulls

!git clone $GIT_REPO bradford_bulls

# Di chuyển vào thư mục v3 (Vì cấu trúc thư mục của bạn có chứa thư mục v3)
%cd /content/bradford_bulls/v3

# Cài đặt các thư viện cần thiết
!pip install -r requirements.txt

# Chuẩn bị thư mục weights
!mkdir -p weights

# Tải mô hình yolo11l.pt. 
# Bạn có thể wget trực tiếp hoặc copy từ Google Drive (nếu bạn đã lưu trên Drive).
!wget -O weights/yolo11l.pt https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11l.pt
# Nếu copy từ drive, bỏ comment dòng dưới:
# !cp /content/drive/MyDrive/Weights/yolo11l.pt weights/

# Chạy script validate kiểm tra GPU và Setup
!python scripts/validate_setup.py

## 3. Chuẩn bị dữ liệu (Video & Metadata)
Copy video từ Google Drive vào không gian chạy của Colab để xử lý nhanh hơn.

In [ ]:
# Tạo thư mục data nếu chưa có
!mkdir -p data/videos
!mkdir -p data/annotation_packages

# COPY video & meta từ Drive vào Colab.
# TODO: Sửa lại đường dẫn /content/drive/... trỏ đúng vào file video của bạn trong Drive
video_drive_path = "/content/drive/MyDrive/RugbyData/match_2026_04_15.mp4"
meta_drive_path = "/content/drive/MyDrive/RugbyData/match_2026_04_15.meta.yaml"

!cp "$video_drive_path" data/videos/
!cp "$meta_drive_path" data/videos/

# Hiển thị để check xem copy thành công chưa
!ls -la data/videos/

## 4. Chạy Pipeline

In [ ]:
# Tên video sau khi copy vào data/videos/ (Thay đổi cho đúng tên)
VIDEO_NAME = "match_2026_04_15"

!python scripts/run_pipeline.py \
    --video data/videos/{VIDEO_NAME}.mp4 \
    --config configs/person_tracking.yaml \
    --output data/annotation_packages/{VIDEO_NAME} \
    --match-meta data/videos/{VIDEO_NAME}.meta.yaml

# LƯU Ý: Nếu bạn KHÔNG có file meta.yaml, hãy truyền explicit kit-context như sau:
# !python scripts/run_pipeline.py \
#     --video data/videos/{VIDEO_NAME}.mp4 \
#     --config configs/person_tracking.yaml \
#     --output data/annotation_packages/{VIDEO_NAME} \
#     --kit-context home

## 5. Zip kết quả và lưu ngược về Google Drive
Do Reviewer UI (Streamlit) không tiện chạy trên Colab, bạn nên zip file output về, sau đó tải về máy local để chạy Streamlit UI review.

In [ ]:
# Nén toàn bộ kết quả
!zip -r {VIDEO_NAME}_package.zip data/annotation_packages/{VIDEO_NAME}

# Copy file Zip về Google Drive
# TODO: Thay đổi thư mục đích trên Drive của bạn
!cp {VIDEO_NAME}_package.zip /content/drive/MyDrive/RugbyData/

print("Đã nén và copy thành công về Google Drive!")